# 04 · Cross-platform positioning
Matched products come from the manual `match_id` column in `data/products.csv`. Search-rank is not captured by the scraper, so that test is not run.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from pricing.db import get_engine, load_snapshots, load_scrape_log
from pricing.clean import clean_snapshots, dim_products, gap_report
from pricing import analysis as an
from pricing.export import load_matches
raw = load_snapshots(get_engine())
clean, report = clean_snapshots(raw)
events = an.discount_events(clean)
print(f"{len(raw):,} raw rows -> {len(clean):,} clean rows, {clean.product_url.nunique()} products")

In [ ]:
import os
pcsv = Path(os.environ.get("PRODUCTS_CSV", "../data/products.csv"))
matches = load_matches(pcsv)
mp = an.matched_products(clean, matches)
pairs = pd.DataFrame(mp["pairs"]); print(len(pairs), "matched pairs")
pairs.sort_values("avg_gap_pct", key=abs, ascending=False).head(15) if len(pairs) else "no matched pairs yet"

In [ ]:
pd.DataFrame(an.unexplained_gaps(mp["pairs"]))

In [ ]:
dim = dim_products(clean).dropna(subset=["avg_rating"])
ax = dim.plot.scatter("avg_price", "avg_rating", s=np.sqrt(dim.latest_reviews.fillna(1)) * 2, c=dim.platform.map({"flipkart": "#2a78d6", "amazon": "#eb6834"}), alpha=.6, figsize=(8, 4.5))
ax.set_title("Value map (bubble = review count)");

In [ ]:
an.response_lag(clean, matches)